In [1]:
import pandas as pd
import re
from hazm import Normalizer, word_tokenize, stopwords_list
import torch  
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

/home/hamid/.pyenv/versions/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df=pd.read_csv('extracted_reviews_v3_final.csv')
df

,index,id,date,clean_text,professor_name,department_name,course,source,attendance,exam_match,...,management,responsiveness,respect,contact_way,term,comment,professor_id,course_normal,course_clean,grading_category
0,3,14,2021-09-05T00:34:39,🧑‍🏫 سید محمد شهرتاش\n🏫 #برق\n📒 بررسی ۱-بررسی۲-...,محمد شهرتاش,برق,بررسی ۱,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,...,['9'],['8'],['10'],چیزی ندارم,مهر 99,چیزی اضافه ایی نیست,1,بررسی ۱-بررسی ۲-حفاظت,بررسی ۱,منصفانه/متوسط
1,3,14,2021-09-05T00:34:39,🧑‍🏫 سید محمد شهرتاش\n🏫 #برق\n📒 بررسی ۱-بررسی۲-...,محمد شهرتاش,برق,بررسی ۲,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,...,['9'],['8'],['10'],چیزی ندارم,مهر 99,چیزی اضافه ایی نیست,1,بررسی ۱-بررسی ۲-حفاظت,بررسی ۲,منصفانه/متوسط
2,3,14,2021-09-05T00:34:39,🧑‍🏫 سید محمد شهرتاش\n🏫 #برق\n📒 بررسی ۱-بررسی۲-...,محمد شهرتاش,برق,حفاظت,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,...,['9'],['8'],['10'],چیزی ندارم,مهر 99,چیزی اضافه ایی نیست,1,بررسی ۱-بررسی ۲-حفاظت,حفاظت,منصفانه/متوسط
3,4,15,2021-09-05T00:35:51,🧑‍🏫 دکتر هاجر فلاحتی\n🏫 #مهندسی_کامپیوتر\n📒 مد...,هاجر فلاحتی,مهندسی_کامپیوتر,مدار منطقی,یادم نمیاد,حضور مهم نیست اما تاثیر مثبت دارد,خیر,...,['3'],['2'],['10'],چیزی ندارم,بهمن 98,خوب درس نمیده اصلا، نمره ها رو خوب نمیده و اصل...,655,مدار منطقی- طراحی سیستم های کامپیوتری,مدار منطقی,سخت‌گیر/بدنمره
4,4,15,2021-09-05T00:35:51,🧑‍🏫 دکتر هاجر فلاحتی\n🏫 #مهندسی_کامپیوتر\n📒 مد...,هاجر فلاحتی,مهندسی_کامپیوتر,طراحی سیستم های کامپیوتری,یادم نمیاد,حضور مهم نیست اما تاثیر مثبت دارد,خیر,...,['3'],['2'],['10'],چیزی ندارم,بهمن 98,خوب درس نمیده اصلا، نمره ها رو خوب نمیده و اصل...,655,مدار منطقی- طراحی سیستم های کامپیوتری,طراحی سیستم های کامپیوتری,سخت‌گیر/بدنمره
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4611,4591,4723,2025-09-27T14:55:48,🧑‍🏫 سید حمیدرضا نوشکاران\n🏫 #ریاضی\n📒 معادلات ...,حمیدرضا نوشکاران,ریاضی,معادلات دیفرانسیل,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم نیست اما تاثیر مثبت دارد,بله,...,['9'],['8'],['10'],ایمیل,بهمن 1403,ترم اولی بود که تدریس می‌کردن ولی تدریس بدی ند...,160,معادلات دیفرانسیل,معادلات دیفرانسیل,منصفانه/متوسط
4612,4592,4724,2025-10-10T14:51:17,🧑‍🏫 عین الله خنجری\n🏫 #مهندسی_کامپیوتر\n📒 پایگ...,عین الله خنجری,مهندسی_کامپیوتر,پایگاه داده پیشرفته,فایل پاورپوینت یا پی دی اف می دهد,حضور و غیاب نمی کند,بله,...,['6'],['1'],['3'],چیزی ندارم,بهمن ۱۴۰۳,تو نمره دادن پروژه‌ها بسیار رندوم و بی‌منطق عم...,690,پایگاه داده پیشرفته,پایگاه داده پیشرفته,سخت‌گیر/بدنمره
4613,4593,4725,2025-10-10T14:51:28,🧑‍🏫 رضا انتظاری ملکی\n🏫 #مهندسی_کامپیوتر\n📒 مب...,رضا انتظاری ملکی,مهندسی_کامپیوتر,مباحث ویژه نرم افزار ۱,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم است و تاثیر مستقیم دارد,بله,...,['10'],['10'],['10'],چیزی ندارم,بهمن ۱۴۰۳,استاد بسیار خوش اخلاق و محترمی هستند. تعداد تم...,666,مباحث ویژه نرم افزار ۱,مباحث ویژه نرم افزار ۱,منصفانه/متوسط
4614,4594,4726,2025-10-10T14:51:31,🧑‍🏫 دکتر بخشایش\n🏫 #مهندسی_مواد\n📒 مواد پیشرفت...,بخشایش,مهندسی_مواد,مواد پیشرفته,فایل پاورپوینت یا پی دی اف می دهد,حضور مهم است و تاثیر مستقیم دارد,بله,...,['10'],['10'],['10'],بله شماره تلفن همراه,بهمن 1401,استاد مطلعی هستند و اطلاعات به روزی دارند.,633,مواد پیشرفته,مواد پیشرفته,منصفانه/متوسط


the following cell does not give me clean data comment col is better than clean_comment !!!!

In [3]:
normalizer = Normalizer()
stopwords = stopwords_list()

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # Normalize Persian text
    text = normalizer.normalize(text)
    # Remove punctuation and non-word characters
    text = re.sub(r'[^\w\s]', '', text)  # Removes punctuation
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords
    tokens = [t for t in tokens if t not in stopwords]
    return " ".join(tokens)

# Apply preprocessing to the 'comment' column
df['cleaned_comment'] = df['comment'].apply(preprocess_text)

In [ ]:
base_model_name = "m3hrdadfi/albert-fa-base-v2"
sentiment_model_name = "m3hrdadfi/albert-fa-base-v2-sentiment-multi"

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
model = AutoModelForSequenceClassification.from_pretrained(sentiment_model_name)

# CORRECT: Remove top_k=None → returns only the TOP label
sentiment_pipeline = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    truncation=True,
    max_length=512,
    device=-1  # CPU,
)

# Test — should now return single dict
print(sentiment_pipeline("عالی بود"))     # → [{'label': 'Positive', 'score': 0.99}]
print(sentiment_pipeline("افتضاح بود"))  # → [{'label': 'Negative', 'score': 0.99}]
print(sentiment_pipeline("معمولی"))      # → [{'label': 'Neutral', 'score': 0.98}]

# Label mapping (human readable)
label_to_score = {
    'Positive':  1.0,
    'Negative': -1.0,
    'Neutral':   0.0
}

def model_sentiment(text):
    if not text.strip():
        return 0.0
    # Now returns list with ONE dict → take [0]
    result = sentiment_pipeline(text)[0]
    label = result['label']
    score = result['score']
    return round(label_to_score[label] * score, 4)

print("Running sentiment analysis on all reviews...")
df['sentiment_score'] = df['cleaned_comment'].apply(model_sentiment)

df['sentiment_label'] = df['sentiment_score'].apply(
    lambda x: 'Positive' if x > 0.2 else ('Negative' if x < -0.2 else 'Neutral')
)

print("All done! First 10 results:")
print(df[['comment', 'sentiment_score', 'sentiment_label']].head(10))

Loading weights: 100%|██████████| 27/27 [00:00<00:00, 124.16it/s, Materializing param=classifier.weight]                                                                


[{'label': 'Positive', 'score': 0.9966872334480286}]
[{'label': 'Negative', 'score': 0.9946186542510986}]
[{'label': 'Neutral', 'score': 0.984097421169281}]
Running sentiment analysis on all reviews...
All done! First 10 results:
                                             comment  sentiment_score  \
0                                چیزی اضافه ایی نیست           0.9584   
1                                چیزی اضافه ایی نیست           0.9584   
2                                چیزی اضافه ایی نیست           0.9584   
3  خوب درس نمیده اصلا، نمره ها رو خوب نمیده و اصل...           0.0000   
4  خوب درس نمیده اصلا، نمره ها رو خوب نمیده و اصل...           0.0000   
5  در کل اگر که دنبال یک استاد با ادب با دانشجو م...           0.0000   
6                                  سخت گیر و پربازده           0.0000   
7                                چیزی اضافه ایی نیست           0.9584   
8  دقت کنید هر جلسه مطالب را بخونید ارشد شب امتحا...           0.7083   
9                                چیزی اض

In [5]:
# Convert all comment to string
df['comment'] = df['comment'].astype(str)

def model_sentiment(text):
    if not isinstance(text, str):
        return 0.0

    text = text.strip()
    if not text:
        return 0.0
    
    result = sentiment_pipeline(text)[0]
    label = result['label']
    score = result['score']
    return round(label_to_score[label] * score, 4)

print("Running sentiment analysis on all reviews...")
df['sentiment_score'] = df['comment'].apply(model_sentiment)

df['sentiment_label'] = df['sentiment_score'].apply(
    lambda x: 'Positive' if x > 0.2 else ('Negative' if x < -0.2 else 'Neutral')
)

print(df[['comment', 'sentiment_score', 'sentiment_label']].head(10))


Running sentiment analysis on all reviews...
                                             comment  sentiment_score  \
0                                چیزی اضافه ایی نیست            0.000   
1                                چیزی اضافه ایی نیست            0.000   
2                                چیزی اضافه ایی نیست            0.000   
3  خوب درس نمیده اصلا، نمره ها رو خوب نمیده و اصل...           -0.715   
4  خوب درس نمیده اصلا، نمره ها رو خوب نمیده و اصل...           -0.715   
5  در کل اگر که دنبال یک استاد با ادب با دانشجو م...            0.000   
6                                  سخت گیر و پربازده            0.000   
7                                چیزی اضافه ایی نیست            0.000   
8  دقت کنید هر جلسه مطالب را بخونید ارشد شب امتحا...            0.000   
9                                چیزی اضافه ایی نیست            0.000   

  sentiment_label  
0         Neutral  
1         Neutral  
2         Neutral  
3        Negative  
4        Negative  
5         Neutral  
6         N

In [ ]:

df.to_csv('public/data/reviews_sentiment_analysis.csv', index=False, encoding='utf-8-sig')
df.to_csv('reviews_sentiment_analysis.csv', index=False, encoding='utf-8-sig')